# Experiment Analysis

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%matplotlib notebook

In [3]:
# Needed to import from the enderscope library
# RUN ONLY ONCE

import os

os.chdir("..")

## Imports

In [4]:
from pathlib import Path
from datetime import datetime as dt

from tqdm import tqdm
from rich.pretty import pprint

import numpy as np
import pandas as pd
import cv2

from matplotlib.figure import Figure

import panel as pn

from enderleaf.const import (
    ImageMergeMode,
    ImageMergeMethod,
    TIME_FORMAT,
    PRECISE_TIME_FORMAT,
    DEFAULT_DATETIME_FORMAT,
    COLOR_SPACES,
)
from enderleaf.draw import image_grid, concat_tile_resize, plot_images_with_histograms
from enderleaf.tools import (
    read_dataframe,
    write_dataframe,
    format_datetime,
    ensure_folder,
)
from enderleaf.image import (
    load_image,
    to_pil,
    canny,
    find_circles,
    filter_circles,
    crop_image,
    Rectangle,
    merge_images,
    merge_images_channels,
    match_previous_rotation,
    get_circles,
    get_channels,
    get_channel,
    equalize_hist,
)
from enderleaf.draw import draw_circles

In [5]:
pn.extension("ipywidgets")

## Constants

In [6]:
EXP = "Exp26DM02"
INOC = "I1"
# PLATE = 0
MONTH = 5
# DAY = 26

PATH_TO_DATA = Path(".").joinpath("output", "job_data", EXP, INOC)
PATH_TO_IMAGES = Path(".").joinpath("output", "images", EXP, INOC)
PATH_TO_PPIMAGES = Path(".").joinpath("output", "pre_processed", EXP, INOC)
MAX_CIRCLES = 3

## Functions

In [7]:
def load(row):
    return load_image(PATH_TO_IMAGES.joinpath(row.file_name))

## Load Data

In [8]:
df = (
    pd.concat([read_dataframe(f) for f in PATH_TO_DATA.glob("*.csv")])
    .sort_values(["plate", "row", "col"])
    .dropna(subset="north")
)
# df = df[df.plate == PLATE]
df = df[df.month == MONTH]
# df = df[df.day == DAY]
df["card_count"] = df[["north", "east", "west", "south"]].astype(int).sum(axis=1)
df["file_ok"] = df["file_name"].apply(lambda x: PATH_TO_IMAGES.joinpath(x).is_file())
print(df[df.file_ok == False].shape)
df = df[df.file_ok == True]
df["file_size"] = df["file_name"].apply(lambda x: PATH_TO_IMAGES.joinpath(x).stat().st_size)
# df = df[df.job_ts != 20260515164717]
df["leaf_id"] = df.plate.astype(str) + df.row.astype(str) + df.col.astype(str)
df

(1425, 42)


,exp,inoc,plate,row,col,date_time,file_name,date,year,month,...,ExposureTime,FocusFoM,GainBlue,GainRed,LensPosition,Lux,card_count,file_ok,file_size,leaf_id
0,Exp26DM02,1,1,1,A,2026-05-22 10:25:40,Exp26DM02#I1#P01#1#A#20260522102540.png,2026-05-22,2026,5,...,23976,11641,0.83,2.4,15.0,831.893005,1,True,5009391,11A
1,Exp26DM02,1,1,1,A,2026-05-22 10:25:41,Exp26DM02#I1#P01#1#A#20260522102541.png,2026-05-22,2026,5,...,23976,9293,0.83,2.4,15.0,685.854980,1,True,5070734,11A
2,Exp26DM02,1,1,1,A,2026-05-22 10:25:41,Exp26DM02#I1#P01#1#A#20260522102541.png,2026-05-22,2026,5,...,23976,6179,0.83,2.4,15.0,476.962402,1,True,5070734,11A
3,Exp26DM02,1,1,1,A,2026-05-22 10:25:42,Exp26DM02#I1#P01#1#A#20260522102542.png,2026-05-22,2026,5,...,23976,9700,0.83,2.4,15.0,625.734375,1,True,5158819,11A
4,Exp26DM02,1,1,1,B,2026-05-22 10:25:45,Exp26DM02#I1#P01#1#B#20260522102545.png,2026-05-22,2026,5,...,23976,12754,0.83,2.4,15.0,907.677307,1,True,5621583,11B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
323,Exp26DM02,I1,P04,9,I,20260527110511051916,Exp26DM02#I1#P04#9#I#e#20260527110511051916.png,2026-05-27,2026,5,...,23976,6978,0.83,2.4,15.0,722.277466,1,True,6970630,P049I
320,Exp26DM02,I1,P04,9,I,20260522180621174861,Exp26DM02#I1#P04#9#I#n#20260522180621174861.png,2026-05-22,2026,5,...,23976,9035,0.83,2.4,15.0,767.907898,1,True,6412533,P049I
321,Exp26DM02,I1,P04,9,I,20260522180622000886,Exp26DM02#I1#P04#9#I#w#20260522180622000886.png,2026-05-22,2026,5,...,23976,8834,0.83,2.4,15.0,737.207703,1,True,6260727,P049I
322,Exp26DM02,I1,P04,9,I,20260522180622704824,Exp26DM02#I1#P04#9#I#s#20260522180622704824.png,2026-05-22,2026,5,...,23976,5130,0.83,2.4,15.0,635.722351,1,True,6321293,P049I


In [ ]:
pd.DataFrame(
    df.groupby(["job_ts", "plate", "light_cycle", "card_count"]).height.mean()
).reset_index().sort_values("plate")

## Select Cycle ID

In [ ]:
sel_date = pn.widgets.Select(
    name="Date",
    options=list(df.date.sort_values().unique()),
    sizing_mode="scale_width",
)
sel_plate = pn.widgets.Select(
    name="Plate",
    options=list(df.plate.unique()),
    sizing_mode="scale_width",
    value="P04",
)
sel_row = pn.widgets.Select(
    name="Row", options=list(df.row.unique()), sizing_mode="scale_width", value=7
)
sel_col = pn.widgets.Select(
    name="Col", options=list(df.col.unique()), sizing_mode="scale_width", value="G"
)

sel_color_space = pn.widgets.Select(
    name="Color space",
    options=["rgb", "hsv", "lab", "yuv", "ycrcb"],
    sizing_mode="scale_width",
    value="rgb",
)
sel_channel_1 = pn.widgets.Select(
    name=COLOR_SPACES[sel_color_space.value][0],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_channel_2 = pn.widgets.Select(
    name=COLOR_SPACES[sel_color_space.value][1],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_channel_3 = pn.widgets.Select(
    name=COLOR_SPACES[sel_color_space.value][2],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_merge_method = pn.widgets.Select(
    name="Merge method",
    options={
        k.name: k
        for k in [
            ImageMergeMethod.RGB,
            ImageMergeMethod.HSV,
            ImageMergeMethod.LAB,
            ImageMergeMethod.YUV,
            ImageMergeMethod.YCrCb,
        ]
    },
    sizing_mode="scale_width",
)
bt_random = pn.widgets.Button(name="Random Disc")

img_out = pn.pane.Matplotlib(sizing_mode="scale_width")


updating = False


def filter_df() -> pd.DataFrame:
    return df[
        (df.date == sel_date.value)
        & (df.plate == sel_plate.value)
        & (df.col == sel_col.value)
        & (df.row == sel_row.value)
    ]


def on_random(event):
    global updating
    updating = True
    try:
        row = df[["date", "plate", "row", "col"]].drop_duplicates().sample(n=1).iloc[0]
        sel_date.value = row.date
        sel_plate.value = row.plate
        sel_row.value = row.row
    finally:
        updating = False
    sel_col.value = row.col


bt_random.on_click(on_random)


@pn.depends(sel_merge_method.param.value, watch=True)
def on_merge_method_changed(mm):
    global updating
    updating = True
    try:
        sel_color_space.value = mm.value[0]
        sel_channel_1.value = mm.value[1][0]
        sel_channel_2.value = mm.value[1][1]
    finally:
        updating = False
    sel_channel_3.value = mm.value[1][2]


@pn.depends(sel_color_space.param.value, watch=True)
def on_color_space_changed(cs):
    sel_channel_1.name = COLOR_SPACES[cs][0]
    sel_channel_2.name = COLOR_SPACES[cs][1]
    sel_channel_3.name = COLOR_SPACES[cs][2]


@pn.depends(
    *[
        w.param.value
        for w in [
            sel_date,
            sel_plate,
            sel_row,
            sel_col,
            sel_color_space,
            sel_channel_1,
            sel_channel_2,
            sel_channel_3,
        ]
    ],
    watch=True,
)
def on_ld_changed(date, plate, row, col, cs, cn1, cn2, cn3):
    if updating is True:
        return
    df_ld = filter_df()
    first_image = load(df_ld.iloc[0])
    height, width, _ = first_image.shape
    circles = get_circles(first_image, color_space="hsv", channel="s", resize_factor=8)
    if len(circles["accepted"]) == 1:
        _, cx, cy, r = circles["accepted"][0]
        crop_data = Rectangle.from_circle((cx, cy, r + 16))
    else:
        crop_data = Rectangle(left=0, top=0, right=width, bottom=height)
    merged_images = []
    card_counts = []
    for card_count in df_ld.card_count.unique():
        image_list = [
            crop_image(load(row[1]), crop_data)
            for row in df_ld[df_ld.card_count == card_count].iterrows()
        ]
        merged_images.append(
            merge_images_channels(
                image_list=image_list, color_space=cs, merge_modes=(cn1, cn2, cn3)
            )
        )
        card_counts.append(f"{card_count} {len(image_list)}")
    img_out.object = plot_images_with_histograms(
        images=merged_images, color_spaces=[cs, "rgb"], titles=card_counts
    )


on_ld_changed(
    sel_date.value,
    sel_plate.value,
    sel_row.value,
    sel_col.value,
    sel_color_space.value,
    sel_channel_1.value,
    sel_channel_2.value,
    sel_channel_3.value,
)

pn.Column(
    pn.Row(
        sel_date,
        sel_plate,
        sel_row,
        sel_col,
        bt_random,
        sel_merge_method,
        sel_color_space,
        sel_channel_1,
        sel_channel_2,
        sel_channel_3,
    ),
    pn.Row(img_out),
)

In [ ]:
df_cc_1 = (
    pd.concat([read_dataframe(f) for f in PATH_TO_DATA.glob("*.csv")])
    .sort_values(["plate", "row", "col"])
    .dropna(subset="north")
)
df_cc_1["card_count"] = (
    df_cc_1[["north", "east", "west", "south"]].astype(int).sum(axis=1)
)
df_cc_1 = df_cc_1[df_cc_1.card_count == 1]
df_cc_1["file_ok"] = df_cc_1["file_name"].apply(
    lambda x: PATH_TO_IMAGES.joinpath(x).is_file()
)
df_cc_1 = df_cc_1[df_cc_1.file_ok == True]
df_cc_1["leaf_id"] = (
    df_cc_1.plate.astype(str) + df_cc_1.row.astype(str) + df_cc_1.col.astype(str)
)
df_cc_1["leaf_pos"] = df_cc_1.row.astype(str) + df_cc_1.col.astype(str)
df_cc_1 = df_cc_1[~df_cc_1.leaf_pos.isin(["1A", "1B", "1C"])]
df_cc_1

In [ ]:
df_cc_1.date.value_counts()

In [ ]:
sel_merge_date = pn.widgets.Select(
    name="Date",
    options=list(df.date.sort_values().unique()),
    sizing_mode="scale_width",
)
sel_merge_plate = pn.widgets.Select(
    name="Plate",
    options=list(df.plate.unique()),
    sizing_mode="scale_width",
    value="P04",
)
sel_merge_row = pn.widgets.Select(
    name="Row", options=list(df.row.unique()), sizing_mode="scale_width", value=7
)
sel_merge_col = pn.widgets.Select(
    name="Col", options=list(df.col.unique()), sizing_mode="scale_width", value="G"
)

bt_random = pn.widgets.Button(name="Random Disc")

img_merge_out = pn.pane.Image(sizing_mode="scale_width")
img_single_merge = pn.pane.Image(sizing_mode="scale_width")
img_channel_merge = pn.pane.Image(sizing_mode="scale_width")
img_diff_merge = pn.pane.Image(sizing_mode="scale_width")


updating = False


def filter_df() -> pd.DataFrame:
    return df_cc_1[
        (df_cc_1.date == sel_merge_date.value)
        & (df_cc_1.plate == sel_merge_plate.value)
        & (df_cc_1.col == sel_merge_col.value)
        & (df_cc_1.row == sel_merge_row.value)
    ]


def on_random(event):
    global updating
    updating = True
    try:
        row = df[["plate", "row", "col"]].drop_duplicates().sample(n=1).iloc[0]

        sel_merge_plate.value = row.plate
        sel_merge_row.value = row.row
    finally:
        updating = False
    sel_merge_col.value = row.col


bt_random.on_click(on_random)

merge_method = ImageMergeMethod.RGB.value


@pn.depends(
    *[w.param.value for w in [sel_merge_date, sel_merge_plate, sel_merge_row, sel_merge_col]],
    watch=True,
)
def on_ld_changed(date, plate, row, col):
    if updating is True:
        return
    df_ld = filter_df()
    first_image = load(df_ld.iloc[0])
    height, width, _ = first_image.shape
    circles = get_circles(first_image, color_space="hsv", channel="s", resize_factor=8)
    if len(circles["accepted"]) == 1:
        _, cx, cy, r = circles["accepted"][0]
        crop_data = Rectangle.from_circle((cx, cy, r + 16))
    else:
        crop_data = Rectangle(left=0, top=0, right=width, bottom=height)
    image_list = [
        crop_image(load(row[1]), crop_data)
        for row in df_ld[df_ld.card_count == 1].iterrows()
    ]
    img_merge_out.object = to_pil(concat_tile_resize([image_list]))
    img_cm = merge_images_channels(
        image_list=image_list, color_space=merge_method[0], merge_modes=merge_method[1]
    )
    img_sm = merge_images(image_list=image_list, merge_mode=ImageMergeMode.MIN)
    img_channel_merge.object = to_pil(img_cm)
    img_single_merge.object = to_pil(img_sm)
    img_diff_merge.object = to_pil(np.abs(img_sm - img_cm))


on_ld_changed(sel_merge_date.value, sel_merge_plate.value, sel_merge_row.value, sel_merge_col.value)

pn.Column(
    pn.Row(sel_merge_date, sel_merge_plate, sel_merge_row, sel_merge_col),
    pn.Row(img_merge_out),
    pn.Row(img_channel_merge, img_single_merge, img_diff_merge),
)

In [ ]:
def add_leaf_disc(
    df_leaf_disc: pd.DataFrame,
    df_target: pd.DataFrame,
    src_image_folder: Path,
    dst_image_folder: Path,
    merge_method=ImageMergeMethod.RGB,
):
    frist_row = df_leaf_disc.iloc[0]
    file_name = f"{frist_row.exp}#I{frist_row.inoc}#P{frist_row.plate}#R{frist_row.row}#C{frist_row.col}#{frist_row.cycle_id}"
    if (
        "file_name" in df_target
        and file_name in df_target.file_name.values
        and dst_image_folder.joinpath(file_name).with_suffix(".png").is_file() is True
    ):
        # Leaf disc has already been successfully handled
        return df_target
    first_image = load(frist_row)
    circles = get_circles(first_image, color_space="hsv", channel="s", resize_factor=8)
    if len(circles["accepted"]) == 1:
        _, cx, cy, r = circles["accepted"][0]
    else:
        print(f"Failed to handle {file_name}")
        return df_target
    crop_data = Rectangle.from_circle((cx, cy, 512))
    image_list = [
        crop_image(load_image(src_image_folder.joinpath(row[1].file_name)), crop_data)
        for row in df_leaf_disc.iterrows()
    ]
    df_means = (
        df_leaf_disc.drop(
            [
                "date_time",
                "file_name",
                "date",
                "year",
                "month",
                "day",
                "time",
                "hour",
                "minute",
                "second",
                "north",
                "east",
                "west",
                "south",
                "light_cycle",
                "center_on_leaf",
                "crop_top",
                "crop_bottom",
                "crop_left",
                "crop_right",
                "card_count",
                "file_ok",
                "leaf_id",
                "leaf_pos",
            ],
            axis=1,
        )
        .groupby(["exp", "inoc", "plate", "row", "col", "cycle_id"])
        .mean()
        .reset_index()
        .assign(
            plate=lambda x: x.plate.astype(str)
            .str.replace("P0", "")
            .str.replace("P", "")
            .astype(int)
        )
        .assign(inoc=lambda x: x.inoc.astype(str).str.replace("I", "").astype(int))
        .assign(file_name=file_name)
    )
    date_time = pd.to_datetime(df_means.cycle_id, format=TIME_FORMAT)
    df_means.insert(6, "second", date_time.dt.second)
    df_means.insert(6, "minute", date_time.dt.minute)
    df_means.insert(6, "hour", date_time.dt.hour)
    df_means.insert(6, "time", date_time.dt.time)
    df_means.insert(6, "day", date_time.dt.day)
    df_means.insert(6, "month", date_time.dt.month)
    df_means.insert(6, "year", date_time.dt.year)
    df_means.insert(6, "date", date_time.dt.date)
    df_means.insert(6, "date_time", date_time)
    if (
        cv2.imwrite(
            str(dst_image_folder.joinpath(file_name).with_suffix(".png")),
            cv2.cvtColor(
                merge_images_channels(
                    image_list=image_list,
                    color_space=merge_method.value[0],
                    merge_modes=merge_method.value[1],
                ),
                cv2.COLOR_RGB2BGR,
            ),
        )
        is True
    ):
        return pd.concat([df_target, df_means])
    else:
        print(f"Failed to write {file_name}")
        return df_target

In [ ]:
ensure_folder(PATH_TO_PPIMAGES)
path_to_csv = PATH_TO_PPIMAGES.parent.joinpath(f"{EXP}_{INOC}").with_suffix(".csv")
df_final = (
    read_dataframe(path_to_csv) if path_to_csv.is_file() is True else pd.DataFrame()
)
for cycle_id in tqdm(df_cc_1.cycle_id.unique()):
    df_final = add_leaf_disc(
        df_leaf_disc=df_cc_1[df_cc_1.cycle_id == cycle_id],
        df_target=df_final,
        dst_image_folder=PATH_TO_PPIMAGES,
        src_image_folder=PATH_TO_IMAGES,
    )
write_dataframe(df_final, path=path_to_csv)

In [ ]:
df_sample = (
    df_final.sample(n=4)
    .assign(plate=lambda x: x.plate.astype(int))
    .sort_values(["date", "plate"])
)

plot_images_with_histograms(
    images=[
        load_image(PATH_TO_PPIMAGES.joinpath(row.file_name).with_suffix(".png"))
        for row in df_sample.itertuples()
    ],
    titles=[
        f"{row.date} - {row.plate}, {row.row}-{row.col} "
        for row in df_sample.itertuples()
    ],
)

In [ ]:
job_ts = df_final.sample(n=1).iloc[0].job_ts
df_plate = df_final[df_final.job_ts == job_ts]

fig = Figure(figsize=(18, 18))
axii = fig.subplots(nrows=9, ncols=9)

columns = df_final.col.sort_values().unique()
rows = df_final.row.sort_values().unique()

pprint(df_plate[["plate", "date"]].sample(n=1))

for c, col in enumerate(columns):
    for r, row in enumerate(rows):
        df_ = df_plate[(df_plate.col == col) & (df_plate.row == row)]
        if len(df_) > 0:
            row = df_.iloc[0]
            axii[c, r].imshow(
                cv2.rotate(
                    load_image(
                        PATH_TO_PPIMAGES.joinpath(row.file_name).with_suffix(".png")
                    ),
                    cv2.ROTATE_90_CLOCKWISE,
                )
            )
        axii[c, r].set_axis_off()

fig.tight_layout()
fig.subplots_adjust(wspace=0, hspace=0)
fig